pip install tree-sitter==0.21.3 tree-sitter-java==0.21.0
python assert_analyzer.py /path/to/tests

In [4]:
!pip3 install tree-sitter==0.21.3 tree-sitter-java==0.21.0

This counts the assert functions used in the LLM generated tests.

In [20]:
"""
Java Assert Analyzer v3 + Batch Runner — single file
Handles .txt files containing Java code with optional markdown code fences.
Includes per-method assertion tracking, method line ranges.
Handles: JUnit 4, JUnit 5, TestNG, AssertJ, Hamcrest

Install:
    pip install tree-sitter tree-sitter-java
"""

import os
import re
import csv
import sys
import json
import inspect as _inspect
from pathlib import Path
from collections import defaultdict
from dataclasses import dataclass, field
from typing import Optional

try:
    import tree_sitter_java as tsjava
    from tree_sitter import Language, Parser
except ImportError:
    sys.exit(
        "Missing deps. Run:\n"
        "  pip install tree-sitter tree-sitter-java"
    )

# ── Handle both old and new tree-sitter APIs ──────────────────────────────────
_lang_params = list(_inspect.signature(Language.__init__).parameters.keys())
_OLD_API = "name" in _lang_params

if _OLD_API:
    JAVA_LANG = Language(tsjava.language(), "java")
    PARSER    = Parser()
    PARSER.set_language(JAVA_LANG)
else:
    JAVA_LANG = Language(tsjava.language())
    PARSER    = Parser(JAVA_LANG)


# =============================================================================
# CONFIGURATION
# =============================================================================

DATASETS = {
    "compiled_pre": "/Volumes/Rachna-HD/ResultsDataset/Exp3LLMOutput/GPT4o/compiled_pre",
    "executed_pre": "/Volumes/Rachna-HD/ResultsDataset/Exp3LLMOutput/GPT4o/execution_pre",
    "detected_bre": "/Volumes/Rachna-HD/ResultsDataset/Exp3LLMOutput/GPT4o/detected_bre",
}

OUTPUT_DIR = "/Volumes/Rachna-HD/AssertAnalysisResults/Exp3LLMOutput/GPT4o"

FILE_PATTERNS = ["*_prompt.txt", "*.java"]


# =============================================================================
# FRAMEWORK DEFINITIONS
# =============================================================================

FRAMEWORKS = {
    "junit4": {
        "imports": ["org.junit.Assert", "org.junit.Assert.*"],
        "static_methods": {
            "assertEquals", "assertNotEquals", "assertTrue", "assertFalse",
            "assertNull", "assertNotNull", "assertSame", "assertNotSame",
            "assertArrayEquals", "assertThat", "fail",
        },
        "qualified_class": "Assert",
    },
    "junit5": {
        "imports": [
            "org.junit.jupiter.api.Assertions",
            "org.junit.jupiter.api.Assertions.*",
        ],
        "static_methods": {
            "assertEquals", "assertNotEquals", "assertTrue", "assertFalse",
            "assertNull", "assertNotNull", "assertSame", "assertNotSame",
            "assertArrayEquals", "assertThrows", "assertDoesNotThrow",
            "assertTimeout", "assertTimeoutPreemptively",
            "assertIterableEquals", "assertLinesMatch", "assertAll", "fail",
        },
        "qualified_class": "Assertions",
    },
    "testng": {
        "imports": [
            "org.testng.Assert",
            "org.testng.Assert.*",
            "org.testng.AssertJUnit",
            "org.testng.AssertJUnit.*",
            "org.testng.asserts.SoftAssert",
        ],
        "static_methods": {
            "assertEquals", "assertNotEquals", "assertTrue", "assertFalse",
            "assertNull", "assertNotNull", "assertSame", "assertNotSame",
            "assertEqualsNoOrder", "assertThrows", "expectThrows", "fail",
        },
        "qualified_class": "Assert",
        "soft_classes": {"SoftAssert"},
    },
    "assertj": {
        "imports": [
            "org.assertj.core.api.Assertions",
            "org.assertj.core.api.Assertions.*",
            "org.assertj.core.api.SoftAssertions",
            "org.assertj.core.api.BDDAssertions",
            "org.assertj.core.api.BDDAssertions.*",
        ],
        "static_methods": {
            "assertThat", "assertThatThrownBy", "assertThatCode",
            "assertThatExceptionOfType", "assertThatNoException",
            "assertThatObject", "assertThatList",
            "catchThrowable", "catchThrowableOfType", "fail",
            "then", "thenThrownBy",
        },
        "qualified_class": "Assertions",
        "fluent": True,
        "soft_classes": {"SoftAssertions", "BDDSoftAssertions", "JUnitSoftAssertions"},
    },
    "hamcrest": {
        "imports": ["org.hamcrest.MatcherAssert", "org.hamcrest.MatcherAssert.*"],
        "static_methods": {"assertThat"},
        "qualified_class": "MatcherAssert",
    },
}

_METHOD_TO_FRAMEWORKS: dict[str, list[str]] = defaultdict(list)
for _fw, _cfg in FRAMEWORKS.items():
    for _m in _cfg.get("static_methods", set()):
        _METHOD_TO_FRAMEWORKS[_m].append(_fw)

ALL_ASSERT_METHODS: set[str] = set(_METHOD_TO_FRAMEWORKS.keys())
ALL_QUALIFIED_CLASSES: set[str] = {
    cfg["qualified_class"] for cfg in FRAMEWORKS.values() if "qualified_class" in cfg
}
ALL_SOFT_CLASSES: set[str] = {
    c for cfg in FRAMEWORKS.values() for c in cfg.get("soft_classes", set())
}
SOFT_CLASS_TO_FRAMEWORK: dict[str, str] = {
    c: fw
    for fw, cfg in FRAMEWORKS.items()
    for c in cfg.get("soft_classes", set())
}


# =============================================================================
# DATA CLASSES
# =============================================================================

@dataclass
class AssertCall:
    method:            str
    line:              int
    source:            str        # 'static' | 'qualified' | 'soft'
    framework:         str
    in_trycatch:       bool = False
    call_text:         str  = ""
    test_method:       str  = ""  # name of the @Test method containing this assert
    test_method_start: int  = 0   # first line of that @Test method
    test_method_end:   int  = 0   # last line of that @Test method


@dataclass
class TestMethod:
    name:         str
    start_line:   int
    end_line:     int
    is_test:      bool = True
    assert_calls: list = field(default_factory=list)


@dataclass
class FileResult:
    path:                   str
    raw_imports:            list[str] = field(default_factory=list)
    frameworks_imported:    set[str]  = field(default_factory=set)
    assert_calls:           list      = field(default_factory=list)
    test_methods:           list      = field(default_factory=list)
    helper_methods:         list[str] = field(default_factory=list)
    has_assert_in_comments: bool      = False
    parse_error:            Optional[str] = None

    @property
    def has_import(self):     return bool(self.frameworks_imported)
    @property
    def has_real_calls(self): return bool(self.assert_calls)
    @property
    def import_only(self):    return self.has_import and not self.has_real_calls
    @property
    def comment_only(self):
        return (
            self.has_assert_in_comments
            and not self.has_real_calls
            and not self.has_import
        )

    def method_counts(self):
        counts = defaultdict(int)
        for c in self.assert_calls:
            counts[c.method] += 1
        return dict(counts)

    def framework_counts(self):
        counts = defaultdict(int)
        for c in self.assert_calls:
            counts[c.framework] += 1
        return dict(counts)


# =============================================================================
# IMPORT ANALYSIS
# =============================================================================

_IMPORT_LINE_RE = re.compile(
    r'^\s*import\s+(?:static\s+)?([a-zA-Z][\w.]*(?:\.\*)?)\s*;', re.MULTILINE
)

def _detect_frameworks_from_imports(raw: str) -> tuple[list[str], set[str]]:
    raw_imports, detected = [], set()
    for m in _IMPORT_LINE_RE.finditer(raw):
        imp = m.group(1)
        raw_imports.append(imp)
        for fw, cfg in FRAMEWORKS.items():
            for pattern in cfg["imports"]:
                if imp == pattern or imp.startswith(pattern.replace(".*", ".")):
                    detected.add(fw)
    return raw_imports, detected


# =============================================================================
# COMMENT SCANNING
# =============================================================================

_COMMENT_RE     = re.compile(r'//[^\n]*|/\*.*?\*/', re.DOTALL)
_ASSERT_WORD_RE = re.compile(r'(?i)\bassert\b')

def _has_assert_in_comments(raw: str) -> bool:
    for m in _COMMENT_RE.finditer(raw):
        if _ASSERT_WORD_RE.search(m.group()):
            return True
    return False


# =============================================================================
# AST HELPERS
# =============================================================================

def _iter_nodes(root):
    """Iteratively yield every node in the subtree (depth-first, pre-order)."""
    stack = [root]
    while stack:
        node = stack.pop()
        yield node
        stack.extend(reversed(node.children))

def _node_text(node, src: bytes) -> str:
    return src[node.start_byte:node.end_byte].decode("utf-8", errors="replace")

def _node_start_line(node) -> int:
    return node.start_point[0] + 1

def _node_end_line(node) -> int:
    return node.end_point[0] + 1

def _is_inside_trycatch(node) -> bool:
    cur = node.parent
    while cur:
        if cur.type == "try_statement":
            return True
        cur = cur.parent
    return False

def _method_name_of(node, src: bytes) -> Optional[str]:
    for child in node.children:
        if child.type == "identifier":
            return _node_text(child, src)
    return None

def _object_name_of(node, src: bytes) -> Optional[str]:
    prev = None
    for child in node.children:
        if child.type == ".":
            break
        prev = child
    if prev and prev.type in ("identifier", "type_identifier"):
        return _node_text(prev, src)
    return None


# =============================================================================
# TEST METHOD EXTRACTOR
# =============================================================================

def _extract_test_methods(root_node, src: bytes) -> list[TestMethod]:
    methods = []
    for node in _iter_nodes(root_node):
        if node.type != "method_declaration":
            continue

        is_test, name = False, ""
        for child in node.children:
            if child.type == "modifiers":
                for mod in child.children:
                    if (mod.type == "marker_annotation"
                            and _node_text(mod, src).lstrip("@") == "Test"):
                        is_test = True
            if child.type == "identifier":
                name = _node_text(child, src)

        methods.append(TestMethod(
            name=name,
            start_line=_node_start_line(node),
            end_line=_node_end_line(node),
            is_test=is_test,
        ))
    return methods


def _find_containing_method(line: int, test_methods: list[TestMethod]) -> Optional[TestMethod]:
    for tm in test_methods:
        if tm.start_line <= line <= tm.end_line:
            return tm
    return None


# =============================================================================
# VARIABLE TYPE TRACKER
# =============================================================================

def _collect_local_var_types(root_node, src: bytes) -> dict[str, str]:
    var_types: dict[str, str] = {}

    for node in _iter_nodes(root_node):
        # Initial declarations: SoftAssertions softly = new SoftAssertions();
        if node.type in ("local_variable_declaration", "field_declaration"):
            type_node = None
            for child in node.children:
                if child.type in ("type_identifier", "generic_type"):
                    type_node = child
                    break
            if type_node:
                type_name = _node_text(type_node, src).split("<")[0].strip()
                for child in node.children:
                    if child.type == "variable_declarator":
                        for sub in child.children:
                            if sub.type == "identifier":
                                var_types[_node_text(sub, src)] = type_name
                                break

        # Reassignments: softly = new SoftAssertions();
        elif node.type == "assignment_expression":
            children = list(node.children)
            if len(children) >= 3:
                lhs = children[0]
                rhs = children[2]
                if lhs.type == "identifier" and rhs.type == "object_creation_expression":
                    for rhs_child in rhs.children:
                        if rhs_child.type in ("type_identifier", "generic_type"):
                            new_type = _node_text(rhs_child, src).split("<")[0].strip()
                            if new_type in ALL_SOFT_CLASSES:
                                var_types[_node_text(lhs, src)] = new_type
                            break

    return var_types


# =============================================================================
# HELPER METHOD DETECTOR
# =============================================================================

def _body_has_assert(method_node, src: bytes) -> bool:
    for node in _iter_nodes(method_node):
        if node.type == "method_invocation":
            name = _method_name_of(node, src)
            if name and name in ALL_ASSERT_METHODS:
                return True
    return False


def _find_helper_method_names(root_node, src: bytes) -> set[str]:
    helpers = set()
    for node in _iter_nodes(root_node):
        if node.type != "method_declaration":
            continue

        is_test = False
        for child in node.children:
            if child.type == "modifiers":
                for mod in child.children:
                    if (mod.type == "marker_annotation"
                            and _node_text(mod, src).lstrip("@") == "Test"):
                        is_test = True

        if not is_test and _body_has_assert(node, src):
            for child in node.children:
                if child.type == "identifier":
                    helpers.add(_node_text(child, src))
                    break
    return helpers


# =============================================================================
# CORE AST WALKER
# =============================================================================

def _collect_assert_calls(
    root_node,
    src: bytes,
    frameworks_imported: set[str],
    var_types: dict[str, str],
    test_methods: list[TestMethod],
) -> list[AssertCall]:

    calls: list[AssertCall] = []

    for node in _iter_nodes(root_node):
        if node.type != "method_invocation":
            continue

        method_name = _method_name_of(node, src)
        obj_name    = _object_name_of(node, src)

        if not method_name:
            continue

        call_text  = _node_text(node, src)
        line       = _node_start_line(node)
        in_try     = _is_inside_trycatch(node)
        containing = _find_containing_method(line, test_methods)
        tm_name    = containing.name       if containing else ""
        tm_start   = containing.start_line if containing else 0
        tm_end     = containing.end_line   if containing else 0
        call       = None

        # A. Static call
        if method_name in ALL_ASSERT_METHODS and obj_name is None:
            fw = _resolve_framework(method_name, frameworks_imported, "static")
            call = AssertCall(
                method=method_name, line=line, source="static", framework=fw,
                in_trycatch=in_try, call_text=call_text[:120],
                test_method=tm_name, test_method_start=tm_start, test_method_end=tm_end,
            )

        # B. Qualified class call (Assert.assertEquals)
        elif method_name in ALL_ASSERT_METHODS and obj_name in ALL_QUALIFIED_CLASSES:
            fw = _resolve_framework(method_name, frameworks_imported, "qualified", obj_name)
            call = AssertCall(
                method=method_name, line=line, source="qualified", framework=fw,
                in_trycatch=in_try, call_text=call_text[:120],
                test_method=tm_name, test_method_start=tm_start, test_method_end=tm_end,
            )

        # C. Fully qualified (org.junit.Assert.assertEquals)
        elif method_name in ALL_ASSERT_METHODS and obj_name is not None:
            if re.search(r'org\.(junit|testng)|org\.assertj|org\.hamcrest', call_text):
                fw = _resolve_framework(method_name, frameworks_imported, "fqn")
                call = AssertCall(
                    method=method_name, line=line, source="qualified", framework=fw,
                    in_trycatch=in_try, call_text=call_text[:120],
                    test_method=tm_name, test_method_start=tm_start, test_method_end=tm_end,
                )

        # D. Soft assertions (softly.assertThat / sa.assertEquals)
        elif obj_name and obj_name in var_types:
            type_name = var_types[obj_name]
            if type_name in ALL_SOFT_CLASSES and method_name in ALL_ASSERT_METHODS:
                fw = SOFT_CLASS_TO_FRAMEWORK.get(type_name, "unknown")
                call = AssertCall(
                    method=method_name, line=line, source="soft", framework=fw,
                    in_trycatch=in_try, call_text=call_text[:120],
                    test_method=tm_name, test_method_start=tm_start, test_method_end=tm_end,
                )

        if call:
            calls.append(call)
            if containing:
                containing.assert_calls.append(call)

    return calls


def _resolve_framework(
    method: str,
    imported: set[str],
    source: str,
    qualifier: Optional[str] = None,
) -> str:
    candidates = _METHOD_TO_FRAMEWORKS.get(method, [])
    matching   = [fw for fw in candidates if fw in imported]

    if len(matching) == 1:
        return matching[0]
    if len(matching) > 1:
        if qualifier == "Assertions":
            for fw in ("junit5", "assertj"):
                if fw in matching: return fw
        if qualifier == "Assert":
            for fw in ("junit4", "testng"):
                if fw in matching: return fw
        return matching[0]
    if candidates:
        return candidates[0]
    return "unknown"


# =============================================================================
# FILE ANALYZER
# =============================================================================

def analyze_file(filepath: str) -> FileResult:
    result = FileResult(path=filepath)

    try:
        raw_bytes = Path(filepath).read_bytes()
        raw_str   = raw_bytes.decode("utf-8", errors="replace")
    except OSError as e:
        result.parse_error = str(e)
        return result

    result.raw_imports, result.frameworks_imported = \
        _detect_frameworks_from_imports(raw_str)
    result.has_assert_in_comments = _has_assert_in_comments(raw_str)

    try:
        tree = PARSER.parse(raw_bytes)
    except Exception as e:
        result.parse_error = f"tree-sitter parse error: {e}"
        return result

    if tree.root_node.has_error:
        result.parse_error = "syntax warnings (partial parse used)"

    root         = tree.root_node
    var_types    = _collect_local_var_types(root, raw_bytes)
    test_methods = _extract_test_methods(root, raw_bytes)
    helper_names = _find_helper_method_names(root, raw_bytes)

    result.test_methods   = test_methods
    result.helper_methods = sorted(helper_names)
    result.assert_calls   = _collect_assert_calls(
        root, raw_bytes,
        result.frameworks_imported,
        var_types,
        test_methods,
    )
    return result


def analyze_directory(root: str, pattern: str = "*Test*.java") -> list[FileResult]:
    return [analyze_file(str(p)) for p in Path(root).rglob(pattern)]


# =============================================================================
# JSON EXPORT
# =============================================================================

def export_json(results: list[FileResult], out_path: str = "ast_analysis.json"):
    output = []
    for r in results:

        # Build set of call ids already attributed to a @Test method
        test_method_call_ids = set()
        method_summary = []
        for tm in r.test_methods:
            if not tm.is_test:
                continue
            for c in tm.assert_calls:
                test_method_call_ids.add(id(c))
            method_summary.append({
                "name":       tm.name,
                "start_line": tm.start_line,
                "end_line":   tm.end_line,
                "assert_calls": [
                    {
                        "method":      c.method,
                        "line":        c.line,
                        "framework":   c.framework,
                        "in_trycatch": c.in_trycatch,
                        "source":      c.source,
                        "call_text":   c.call_text,
                    }
                    for c in tm.assert_calls
                ],
            })

        # Flat call list — tag calls not owned by any @Test as coming from helpers
        flat_calls = []
        for c in r.assert_calls:
            flat_calls.append({
                "method":            c.method,
                "line":              c.line,
                "framework":         c.framework,
                "in_trycatch":       c.in_trycatch,
                "source":            c.source,
                "call_text":         c.call_text,
                "test_method":       c.test_method,
                "test_method_start": c.test_method_start,
                "test_method_end":   c.test_method_end,
                "in_test_method":    id(c) in test_method_call_ids,
            })

        output.append({
            "file":                   Path(r.path).name,
            "path":                   r.path,
            "frameworks_imported":    list(r.frameworks_imported),
            "has_import":             r.has_import,
            "has_real_calls":         r.has_real_calls,
            "import_only":            r.import_only,
            "comment_only":           r.comment_only,
            "has_assert_in_comments": r.has_assert_in_comments,
            "total_assert_calls":     len(r.assert_calls),
            "helper_methods":         r.helper_methods,
            "parse_error":            r.parse_error,
            "assert_calls":           flat_calls,
            "test_methods":           method_summary,
        })

    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(output, f, indent=2)
    print(f"  Details: {out_path}")


# =============================================================================
# PREPROCESSING (.txt → .java)
# =============================================================================

def clean_java_code(content: str) -> str:
    # Remove opening fence (```java or ```)
    content = re.sub(r'^```java\s*\n', '', content, flags=re.MULTILINE)
    content = re.sub(r'^```\s*\n',     '', content, flags=re.MULTILINE)
    # Remove closing fence
    content = re.sub(r'\n```\s*$',     '', content, flags=re.MULTILINE)
    content = re.sub(r'^```\s*$',      '', content, flags=re.MULTILINE)
    # Strip any surrounding non-Java text before the first Java-looking line
    content = re.sub(r'^.*?(?=\s*(package|import|public|class|//|/\*))',
                     '', content, count=1, flags=re.DOTALL)
    return content.strip()


def extract_class_name(content: str) -> Optional[str]:
    m = re.search(r'public\s+class\s+(\w+)', content)
    return m.group(1) if m else None


def prepare_java_file(txt_path: Path) -> Optional[Path]:
    try:
        content = txt_path.read_text(encoding='utf-8')
    except Exception as e:
        print(f"    ⚠️  Error reading {txt_path.name}: {e}")
        return None

    cleaned    = clean_java_code(content)
    class_name = extract_class_name(cleaned) or txt_path.stem.replace('_prompt', '')

    java_dir  = txt_path.parent / "_java_files"
    java_dir.mkdir(exist_ok=True)
    java_file = java_dir / f"{class_name}.java"

    try:
        java_file.write_text(cleaned, encoding='utf-8')
        return java_file
    except Exception as e:
        print(f"   Error writing {java_file}: {e}")
        return None


def find_test_files(root_dir: str, patterns: list[str]) -> list[Path]:
    files = []
    root  = Path(root_dir)
    for pattern in patterns:
        for f in root.rglob(pattern):
            if "_java_files" not in f.parts:
                files.append(f)
    return sorted(set(files))


# =============================================================================
# STATS + REPORTING
# =============================================================================

def compute_stats(results: list) -> dict:
    total = len(results)

    # ── File-level — mutually exclusive categories that sum to total ──────────
    files_with_calls  = sum(1 for r in results if r.has_real_calls)
    files_import_only = sum(1 for r in results if r.import_only)
    files_comment_only = sum(1 for r in results if r.comment_only)
    files_no_assert   = sum(
        1 for r in results
        if not r.has_real_calls and not r.import_only and not r.comment_only
    )
    # Sanity: files_with_calls + files_import_only + files_comment_only + files_no_assert == total

    files_with_helpers = sum(1 for r in results if r.helper_methods)

    # ── Framework breakdown: both calls and files ─────────────────────────────
    framework_call_counts:  dict[str, int] = defaultdict(int)
    framework_file_counts:  dict[str, int] = defaultdict(int)
    method_counts:          dict[str, int] = defaultdict(int)

    for r in results:
        frameworks_seen_in_file = set()
        for c in r.assert_calls:
            framework_call_counts[c.framework] += 1
            method_counts[c.method]            += 1
            frameworks_seen_in_file.add(c.framework)
        for fw in frameworks_seen_in_file:
            framework_file_counts[fw] += 1

    # ── @Test method level ────────────────────────────────────────────────────
    all_test_methods = [
        tm for r in results
        for tm in r.test_methods if tm.is_test
    ]
    total_test_methods = len(all_test_methods)

    test_assert_counts   = [len(tm.assert_calls) for tm in all_test_methods]
    methods_with_asserts = sum(1 for c in test_assert_counts if c > 0)
    methods_no_asserts   = total_test_methods - methods_with_asserts
    total_calls          = sum(test_assert_counts)
    min_asserts          = min(test_assert_counts) if test_assert_counts else 0
    max_asserts          = max(test_assert_counts) if test_assert_counts else 0
    avg_asserts          = total_calls / total_test_methods if total_test_methods > 0 else 0

    in_trycatch = sum(
        sum(1 for c in r.assert_calls if c.in_trycatch) for r in results
    )
    files_with_trycatch = sum(
        1 for r in results if any(c.in_trycatch for c in r.assert_calls)
    )
    soft = sum(1 for r in results if any(c.source == "soft" for c in r.assert_calls))

    return {
        # File-level (mutually exclusive, sums to total_files)
        "total_files":                  total,
        "files_with_assert_calls":      files_with_calls,
        "files_import_only":            files_import_only,
        "files_comment_only":           files_comment_only,
        "files_no_assert":              files_no_assert,
        "files_with_helpers":           files_with_helpers,
        # @Test method-level
        "total_test_methods":           total_test_methods,
        "test_methods_with_asserts":    methods_with_asserts,
        "test_methods_without_asserts": methods_no_asserts,
        "total_assert_calls":           total_calls,
        "min_asserts_per_test_method":  min_asserts,
        "max_asserts_per_test_method":  max_asserts,
        "avg_asserts_per_test_method":  avg_asserts,
        "calls_in_trycatch":            in_trycatch,
        "files_with_trycatch_asserts":  files_with_trycatch,
        "files_with_soft_assertions":   soft,
        # Breakdowns — calls vs files clearly separated
        "framework_call_counts":        dict(framework_call_counts),
        "framework_file_counts":        dict(framework_file_counts),
        "method_breakdown":             dict(method_counts),
    }


def print_summary(stats: dict):
    total = stats['total_files']

    print(f"\n  📊 SUMMARY")
    print(f"  {'-'*66}")

    print(f"\n  FILE LEVEL  (total: {total})")
    print(f"  {'Category':<40} {'Count':>6}  {'%':>6}")
    print(f"  {'-'*55}")
    categories = [
        ("Files with assert calls",   "files_with_assert_calls"),
        ("Files with import only",    "files_import_only"),
        ("Files with comments only",  "files_comment_only"),
        ("Files with no assert",      "files_no_assert"),
    ]
    for label, key in categories:
        cnt = stats[key]
        pct = 100 * cnt / total if total > 0 else 0
        print(f"  {label:<40} {cnt:>6}  {pct:>5.1f}%")
    # Confirm they sum
    subtotal = sum(stats[k] for _, k in categories)
    print(f"  {'-'*55}")
    print(f"  {'Total':<40} {subtotal:>6}")
    print(f"  Files with helper methods       : {stats['files_with_helpers']}")

    print(f"\n  @TEST METHOD LEVEL")
    print(f"  Total @Test methods             : {stats['total_test_methods']}")
    print(f"  @Test methods with asserts      : {stats['test_methods_with_asserts']}")
    print(f"  @Test methods without asserts   : {stats['test_methods_without_asserts']}")
    print(f"  ")
    print(f"  Total assert calls              : {stats['total_assert_calls']}")
    print(f"  Min asserts per @Test method    : {stats['min_asserts_per_test_method']}")
    print(f"  Max asserts per @Test method    : {stats['max_asserts_per_test_method']}")
    print(f"  Avg asserts per @Test method    : {stats['avg_asserts_per_test_method']:.2f}")
    print(f"  Calls inside try/catch          : {stats['calls_in_trycatch']}")
    print(f"  Files with assert in try/catch  : {stats['files_with_trycatch_asserts']}")
    print(f"  Files with soft assertions      : {stats['files_with_soft_assertions']}")

    if stats['framework_call_counts']:
        print(f"\n  Framework breakdown (calls | files using it):")
        all_fw = sorted(
            set(stats['framework_call_counts']) | set(stats['framework_file_counts'])
        )
        for fw in sorted(all_fw, key=lambda x: -stats['framework_call_counts'].get(x, 0)):
            calls = stats['framework_call_counts'].get(fw, 0)
            files = stats['framework_file_counts'].get(fw, 0)
            print(f"    {fw:<20} {calls:>5} calls   {files:>5} files")

    if stats['method_breakdown']:
        print(f"\n  Top 10 assert methods:")
        for m, cnt in sorted(stats['method_breakdown'].items(), key=lambda x: -x[1])[:10]:
            print(f"    {m:<30} {cnt:>5} calls")
    print()


def generate_comparison_report(all_results: dict):
    print(f"\n{'='*70}")
    print(f"  COMPARISON ACROSS DATASETS")
    print(f"{'='*70}\n")

    rows = []
    for name in ["compiled_pre", "executed_pre", "detected_bre"]:
        if name not in all_results or not all_results[name]:
            continue
        s = all_results[name]["stats"]
        rows.append({
            "Dataset":       name,
            "Files":         s["total_files"],
            "w/ Calls":      s["files_with_assert_calls"],
            "Import Only":   s["files_import_only"],
            "No Assert":     s["files_no_assert"],
            "@Test Methods": s["total_test_methods"],
            "w/ Asserts":    s["test_methods_with_asserts"],
            "w/o Asserts":   s["test_methods_without_asserts"],
            "Total Calls":   s["total_assert_calls"],
            "Min":           s["min_asserts_per_test_method"],
            "Max":           s["max_asserts_per_test_method"],
            "Avg":           f"{s['avg_asserts_per_test_method']:.2f}",
            "In Try/Catch":  s["calls_in_trycatch"],
            "Files w/ Try/Catch": s["files_with_trycatch_asserts"],
        })

    if not rows:
        print("  No data to compare.\n")
        return

    headers = list(rows[0].keys())
    widths  = [
        max(len(str(r.get(h, ""))) for r in rows + [{h: h}])
        for h in headers
    ]
    print("  " + " | ".join(h.ljust(w) for h, w in zip(headers, widths)))
    print("  " + "-+-".join("-" * w for w in widths))
    for row in rows:
        print("  " + " | ".join(str(row.get(h, "")).ljust(w) for h, w in zip(headers, widths)))
    print()


# =============================================================================
# DATASET RUNNER
# =============================================================================

def analyze_dataset(name: str, root_dir: str) -> Optional[dict]:
    print(f"\n{'='*70}")
    print(f"  Analyzing : {name}")
    print(f"  Location  : {root_dir}")
    print(f"{'='*70}\n")

    if not Path(root_dir).exists():
        print(f"    Directory does not exist, skipping.\n")
        return None

    files = find_test_files(root_dir, FILE_PATTERNS)
    if not files:
        print(f"    No test files found.\n")
        return None

    print(f"  Found {len(files)} source files")
    print(f"  Preparing Java files...\n")

    java_files = []
    for f in files:
        if f.suffix == '.txt':
            jf = prepare_java_file(f)
            if jf:
                java_files.append(jf)
        else:
            java_files.append(f)

    if not java_files:
        print(f"    No valid Java files after preprocessing.\n")
        return None

    print(f"  Analyzing {len(java_files)} Java files...\n")

    results = []
    for i, filepath in enumerate(java_files, 1):
        if i % 50 == 0:
            print(f"    Progress: {i}/{len(java_files)}")
        results.append(analyze_file(str(filepath)))

    # Guard: ensure every result has test_methods populated
    for r in results:
        if not hasattr(r, 'test_methods') or r.test_methods is None:
            r.test_methods = []

    stats = compute_stats(results)
    stats.update({
        "dataset_name":   name,
        "root_dir":       root_dir,
        "file_count":     len(files),
        "analyzed_count": len(java_files),
    })

    print_summary(stats)
    return {"stats": stats, "results": results}


def save_json_reports(all_results: dict):
    output_dir = Path(OUTPUT_DIR)
    output_dir.mkdir(parents=True, exist_ok=True)

    for name, data in all_results.items():
        if not data:
            continue
        stats_file = output_dir / f"{name}_stats.json"
        with open(stats_file, "w") as f:
            json.dump(data["stats"], f, indent=2)
        print(f"  Stats  : {stats_file}")

        details_file = str(output_dir / f"{name}_details.json")
        export_json(data["results"], details_file)

    print()


# =============================================================================
# MAIN
# =============================================================================

def main():
    print(f"\n{'='*70}")
    print(f"  Java Assert Analysis Runner  (v3 — per-method tracking)")
    print(f"{'='*70}")

    all_results = {}
    for name, path in DATASETS.items():
        result = analyze_dataset(name, path)
        if result:
            all_results[name] = result

    if not all_results:
        print("\n  No datasets analyzed. Check your paths.\n")
        return

    generate_comparison_report(all_results)
    save_json_reports(all_results)

    print(f"\n{'='*70}")
    print(f"   Analysis complete!")
    print(f"   Reports saved to : {OUTPUT_DIR}")
    print(f"   Cleaned .java    : each instance/_java_files/")
    print(f"{'='*70}\n")


if __name__ == "__main__":
    main()


  Java Assert Analysis Runner  (v3 — per-method tracking)

  Analyzing : compiled_pre
  Location  : /Volumes/Rachna-HD/ResultsDataset/Exp3LLMOutput/GPT4o/compiled_pre

  Found 971 source files
  Preparing Java files...

  Analyzing 971 Java files...

    Progress: 50/971
    Progress: 100/971
    Progress: 150/971
    Progress: 200/971
    Progress: 250/971
    Progress: 300/971
    Progress: 350/971
    Progress: 400/971
    Progress: 450/971
    Progress: 500/971
    Progress: 550/971
    Progress: 600/971
    Progress: 650/971
    Progress: 700/971
    Progress: 750/971
    Progress: 800/971
    Progress: 850/971
    Progress: 900/971
    Progress: 950/971

  📊 SUMMARY
  ------------------------------------------------------------------

  FILE LEVEL  (total: 971)
  Category                                  Count       %
  -------------------------------------------------------
  Files with assert calls                     904   93.1%
  Files with import only                       

For Sankey : https://sankeymatic.com/build/

In [21]:
all_results = main()


  Java Assert Analysis Runner  (v3 — per-method tracking)

  Analyzing : compiled_pre
  Location  : /Volumes/Rachna-HD/ResultsDataset/Exp3LLMOutput/GPT4o/compiled_pre

  Found 971 source files
  Preparing Java files...

  Analyzing 971 Java files...

    Progress: 50/971
    Progress: 100/971
    Progress: 150/971
    Progress: 200/971
    Progress: 250/971
    Progress: 300/971
    Progress: 350/971
    Progress: 400/971
    Progress: 450/971
    Progress: 500/971
    Progress: 550/971
    Progress: 600/971
    Progress: 650/971
    Progress: 700/971
    Progress: 750/971
    Progress: 800/971
    Progress: 850/971
    Progress: 900/971
    Progress: 950/971

  📊 SUMMARY
  ------------------------------------------------------------------

  FILE LEVEL  (total: 971)
  Category                                  Count       %
  -------------------------------------------------------
  Files with assert calls                     904   93.1%
  Files with import only                       

In [26]:
"""
Java Assert Analyzer v3 + Batch Runner — single file
Handles .txt files containing Java code with optional markdown code fences.
Includes per-method assertion tracking, method line ranges.
Handles: JUnit 4, JUnit 5, TestNG, AssertJ, Hamcrest

Install:
    pip install tree-sitter tree-sitter-java
"""

import os
import re
import csv
import sys
import json
import inspect as _inspect
from pathlib import Path
from collections import defaultdict
from dataclasses import dataclass, field
from typing import Optional

try:
    import tree_sitter_java as tsjava
    from tree_sitter import Language, Parser
except ImportError:
    sys.exit(
        "Missing deps. Run:\n"
        "  pip install tree-sitter tree-sitter-java"
    )

# ── Handle both old and new tree-sitter APIs ──────────────────────────────────
_lang_params = list(_inspect.signature(Language.__init__).parameters.keys())
_OLD_API = "name" in _lang_params

if _OLD_API:
    JAVA_LANG = Language(tsjava.language(), "java")
    PARSER    = Parser()
    PARSER.set_language(JAVA_LANG)
else:
    JAVA_LANG = Language(tsjava.language())
    PARSER    = Parser(JAVA_LANG)


# =============================================================================
# CONFIGURATION
# =============================================================================

DATASETS = {
    "compiled_pre": "/Volumes/Rachna-HD/ResultsDataset/Exp3LLMOutput/GPT4o/compiled_pre",
    "executed_pre": "/Volumes/Rachna-HD/ResultsDataset/Exp3LLMOutput/GPT4o/execution_pre",
    "detected_bre": "/Volumes/Rachna-HD/ResultsDataset/Exp3LLMOutput/GPT4o/detected_bre",
}

OUTPUT_DIR = "/Volumes/Rachna-HD/AssertAnalysisResults/Exp3LLMOutput/GPT4o"

FILE_PATTERNS = ["*_prompt.txt", "*.java"]

RESULTS_CACHE = f"{OUTPUT_DIR}/all_results_cache.json"


# =============================================================================
# FRAMEWORK DEFINITIONS
# =============================================================================

FRAMEWORKS = {
    "junit4": {
        "imports": ["org.junit.Assert", "org.junit.Assert.*"],
        "static_methods": {
            "assertEquals", "assertNotEquals", "assertTrue", "assertFalse",
            "assertNull", "assertNotNull", "assertSame", "assertNotSame",
            "assertArrayEquals", "assertThat", "fail",
        },
        "qualified_class": "Assert",
    },
    "junit5": {
        "imports": [
            "org.junit.jupiter.api.Assertions",
            "org.junit.jupiter.api.Assertions.*",
        ],
        "static_methods": {
            "assertEquals", "assertNotEquals", "assertTrue", "assertFalse",
            "assertNull", "assertNotNull", "assertSame", "assertNotSame",
            "assertArrayEquals", "assertThrows", "assertDoesNotThrow",
            "assertTimeout", "assertTimeoutPreemptively",
            "assertIterableEquals", "assertLinesMatch", "assertAll", "fail",
        },
        "qualified_class": "Assertions",
    },
    "testng": {
        "imports": [
            "org.testng.Assert",
            "org.testng.Assert.*",
            "org.testng.AssertJUnit",
            "org.testng.AssertJUnit.*",
            "org.testng.asserts.SoftAssert",
        ],
        "static_methods": {
            "assertEquals", "assertNotEquals", "assertTrue", "assertFalse",
            "assertNull", "assertNotNull", "assertSame", "assertNotSame",
            "assertEqualsNoOrder", "assertThrows", "expectThrows", "fail",
        },
        "qualified_class": "Assert",
        "soft_classes": {"SoftAssert"},
    },
    "assertj": {
        "imports": [
            "org.assertj.core.api.Assertions",
            "org.assertj.core.api.Assertions.*",
            "org.assertj.core.api.SoftAssertions",
            "org.assertj.core.api.BDDAssertions",
            "org.assertj.core.api.BDDAssertions.*",
        ],
        "static_methods": {
            "assertThat", "assertThatThrownBy", "assertThatCode",
            "assertThatExceptionOfType", "assertThatNoException",
            "assertThatObject", "assertThatList",
            "catchThrowable", "catchThrowableOfType", "fail",
            "then", "thenThrownBy",
        },
        "qualified_class": "Assertions",
        "fluent": True,
        "soft_classes": {"SoftAssertions", "BDDSoftAssertions", "JUnitSoftAssertions"},
    },
    "hamcrest": {
        "imports": ["org.hamcrest.MatcherAssert", "org.hamcrest.MatcherAssert.*"],
        "static_methods": {"assertThat"},
        "qualified_class": "MatcherAssert",
    },
}

_METHOD_TO_FRAMEWORKS: dict[str, list[str]] = defaultdict(list)
for _fw, _cfg in FRAMEWORKS.items():
    for _m in _cfg.get("static_methods", set()):
        _METHOD_TO_FRAMEWORKS[_m].append(_fw)

ALL_ASSERT_METHODS: set[str] = set(_METHOD_TO_FRAMEWORKS.keys())
ALL_QUALIFIED_CLASSES: set[str] = {
    cfg["qualified_class"] for cfg in FRAMEWORKS.values() if "qualified_class" in cfg
}
ALL_SOFT_CLASSES: set[str] = {
    c for cfg in FRAMEWORKS.values() for c in cfg.get("soft_classes", set())
}
SOFT_CLASS_TO_FRAMEWORK: dict[str, str] = {
    c: fw
    for fw, cfg in FRAMEWORKS.items()
    for c in cfg.get("soft_classes", set())
}


# =============================================================================
# DATA CLASSES
# =============================================================================

@dataclass
class AssertCall:
    method:            str
    line:              int
    source:            str        # 'static' | 'qualified' | 'soft'
    framework:         str
    in_trycatch:       bool = False
    call_text:         str  = ""
    test_method:       str  = ""
    test_method_start: int  = 0
    test_method_end:   int  = 0


@dataclass
class TestMethod:
    name:         str
    start_line:   int
    end_line:     int
    is_test:      bool = True
    assert_calls: list = field(default_factory=list)


@dataclass
class FileResult:
    path:                   str
    raw_imports:            list[str] = field(default_factory=list)
    frameworks_imported:    set[str]  = field(default_factory=set)
    assert_calls:           list      = field(default_factory=list)
    test_methods:           list      = field(default_factory=list)
    helper_methods:         list[str] = field(default_factory=list)
    has_assert_in_comments: bool      = False
    parse_error:            Optional[str] = None

    @property
    def has_import(self):     return bool(self.frameworks_imported)
    @property
    def has_real_calls(self): return bool(self.assert_calls)
    @property
    def import_only(self):    return self.has_import and not self.has_real_calls
    @property
    def comment_only(self):
        return (
            self.has_assert_in_comments
            and not self.has_real_calls
            and not self.has_import
        )

    def method_counts(self):
        counts = defaultdict(int)
        for c in self.assert_calls:
            counts[c.method] += 1
        return dict(counts)

    def framework_counts(self):
        counts = defaultdict(int)
        for c in self.assert_calls:
            counts[c.framework] += 1
        return dict(counts)


# =============================================================================
# IMPORT ANALYSIS
# =============================================================================

_IMPORT_LINE_RE = re.compile(
    r'^\s*import\s+(?:static\s+)?([a-zA-Z][\w.]*(?:\.\*)?)\s*;', re.MULTILINE
)

def _detect_frameworks_from_imports(raw: str) -> tuple[list[str], set[str]]:
    raw_imports, detected = [], set()
    for m in _IMPORT_LINE_RE.finditer(raw):
        imp = m.group(1)
        raw_imports.append(imp)
        for fw, cfg in FRAMEWORKS.items():
            for pattern in cfg["imports"]:
                if imp == pattern or imp.startswith(pattern.replace(".*", ".")):
                    detected.add(fw)
    return raw_imports, detected


# =============================================================================
# COMMENT SCANNING
# =============================================================================

_COMMENT_RE     = re.compile(r'//[^\n]*|/\*.*?\*/', re.DOTALL)
_ASSERT_WORD_RE = re.compile(r'(?i)\bassert\b')

def _has_assert_in_comments(raw: str) -> bool:
    for m in _COMMENT_RE.finditer(raw):
        if _ASSERT_WORD_RE.search(m.group()):
            return True
    return False


# =============================================================================
# AST HELPERS
# =============================================================================

def _iter_nodes(root):
    """Iteratively yield every node in the subtree (depth-first, pre-order)."""
    stack = [root]
    while stack:
        node = stack.pop()
        yield node
        stack.extend(reversed(node.children))

def _node_text(node, src: bytes) -> str:
    return src[node.start_byte:node.end_byte].decode("utf-8", errors="replace")

def _node_start_line(node) -> int:
    return node.start_point[0] + 1

def _node_end_line(node) -> int:
    return node.end_point[0] + 1

def _is_inside_trycatch(node) -> bool:
    cur = node.parent
    while cur:
        if cur.type == "try_statement":
            return True
        cur = cur.parent
    return False

def _method_name_of(node, src: bytes) -> Optional[str]:
    for child in node.children:
        if child.type == "identifier":
            return _node_text(child, src)
    return None

def _object_name_of(node, src: bytes) -> Optional[str]:
    prev = None
    for child in node.children:
        if child.type == ".":
            break
        prev = child
    if prev and prev.type in ("identifier", "type_identifier"):
        return _node_text(prev, src)
    return None


# =============================================================================
# TEST METHOD EXTRACTOR
# =============================================================================

def _extract_test_methods(root_node, src: bytes) -> list[TestMethod]:
    methods = []
    for node in _iter_nodes(root_node):
        if node.type != "method_declaration":
            continue

        is_test, name = False, ""
        for child in node.children:
            if child.type == "modifiers":
                for mod in child.children:
                    if (mod.type == "marker_annotation"
                            and _node_text(mod, src).lstrip("@") == "Test"):
                        is_test = True
            if child.type == "identifier":
                name = _node_text(child, src)

        methods.append(TestMethod(
            name=name,
            start_line=_node_start_line(node),
            end_line=_node_end_line(node),
            is_test=is_test,
        ))
    return methods


def _find_containing_method(line: int, test_methods: list[TestMethod]) -> Optional[TestMethod]:
    for tm in test_methods:
        if tm.start_line <= line <= tm.end_line:
            return tm
    return None


# =============================================================================
# VARIABLE TYPE TRACKER
# =============================================================================

def _collect_local_var_types(root_node, src: bytes) -> dict[str, str]:
    var_types: dict[str, str] = {}

    for node in _iter_nodes(root_node):
        if node.type in ("local_variable_declaration", "field_declaration"):
            type_node = None
            for child in node.children:
                if child.type in ("type_identifier", "generic_type"):
                    type_node = child
                    break
            if type_node:
                type_name = _node_text(type_node, src).split("<")[0].strip()
                for child in node.children:
                    if child.type == "variable_declarator":
                        for sub in child.children:
                            if sub.type == "identifier":
                                var_types[_node_text(sub, src)] = type_name
                                break

        elif node.type == "assignment_expression":
            children = list(node.children)
            if len(children) >= 3:
                lhs = children[0]
                rhs = children[2]
                if lhs.type == "identifier" and rhs.type == "object_creation_expression":
                    for rhs_child in rhs.children:
                        if rhs_child.type in ("type_identifier", "generic_type"):
                            new_type = _node_text(rhs_child, src).split("<")[0].strip()
                            if new_type in ALL_SOFT_CLASSES:
                                var_types[_node_text(lhs, src)] = new_type
                            break

    return var_types


# =============================================================================
# HELPER METHOD DETECTOR
# =============================================================================

def _body_has_assert(method_node, src: bytes) -> bool:
    for node in _iter_nodes(method_node):
        if node.type == "method_invocation":
            name = _method_name_of(node, src)
            if name and name in ALL_ASSERT_METHODS:
                return True
    return False


def _find_helper_method_names(root_node, src: bytes) -> set[str]:
    helpers = set()
    for node in _iter_nodes(root_node):
        if node.type != "method_declaration":
            continue

        is_test = False
        for child in node.children:
            if child.type == "modifiers":
                for mod in child.children:
                    if (mod.type == "marker_annotation"
                            and _node_text(mod, src).lstrip("@") == "Test"):
                        is_test = True

        if not is_test and _body_has_assert(node, src):
            for child in node.children:
                if child.type == "identifier":
                    helpers.add(_node_text(child, src))
                    break
    return helpers


# =============================================================================
# CORE AST WALKER
# =============================================================================

def _collect_assert_calls(
    root_node,
    src: bytes,
    frameworks_imported: set[str],
    var_types: dict[str, str],
    test_methods: list[TestMethod],
) -> list[AssertCall]:

    calls: list[AssertCall] = []

    for node in _iter_nodes(root_node):
        if node.type != "method_invocation":
            continue

        method_name = _method_name_of(node, src)
        obj_name    = _object_name_of(node, src)

        if not method_name:
            continue

        call_text  = _node_text(node, src)
        line       = _node_start_line(node)
        in_try     = _is_inside_trycatch(node)
        containing = _find_containing_method(line, test_methods)
        tm_name    = containing.name       if containing else ""
        tm_start   = containing.start_line if containing else 0
        tm_end     = containing.end_line   if containing else 0
        call       = None

        # A. Static call
        if method_name in ALL_ASSERT_METHODS and obj_name is None:
            fw = _resolve_framework(method_name, frameworks_imported, "static")
            call = AssertCall(
                method=method_name, line=line, source="static", framework=fw,
                in_trycatch=in_try, call_text=call_text[:120],
                test_method=tm_name, test_method_start=tm_start, test_method_end=tm_end,
            )

        # B. Qualified class call (Assert.assertEquals)
        elif method_name in ALL_ASSERT_METHODS and obj_name in ALL_QUALIFIED_CLASSES:
            fw = _resolve_framework(method_name, frameworks_imported, "qualified", obj_name)
            call = AssertCall(
                method=method_name, line=line, source="qualified", framework=fw,
                in_trycatch=in_try, call_text=call_text[:120],
                test_method=tm_name, test_method_start=tm_start, test_method_end=tm_end,
            )

        # C. Fully qualified (org.junit.Assert.assertEquals)
        elif method_name in ALL_ASSERT_METHODS and obj_name is not None:
            if re.search(r'org\.(junit|testng)|org\.assertj|org\.hamcrest', call_text):
                fw = _resolve_framework(method_name, frameworks_imported, "fqn")
                call = AssertCall(
                    method=method_name, line=line, source="qualified", framework=fw,
                    in_trycatch=in_try, call_text=call_text[:120],
                    test_method=tm_name, test_method_start=tm_start, test_method_end=tm_end,
                )

        # D. Soft assertions (softly.assertThat / sa.assertEquals)
        elif obj_name and obj_name in var_types:
            type_name = var_types[obj_name]
            if type_name in ALL_SOFT_CLASSES and method_name in ALL_ASSERT_METHODS:
                fw = SOFT_CLASS_TO_FRAMEWORK.get(type_name, "unknown")
                call = AssertCall(
                    method=method_name, line=line, source="soft", framework=fw,
                    in_trycatch=in_try, call_text=call_text[:120],
                    test_method=tm_name, test_method_start=tm_start, test_method_end=tm_end,
                )

        if call:
            calls.append(call)
            if containing:
                containing.assert_calls.append(call)

    return calls


def _resolve_framework(
    method: str,
    imported: set[str],
    source: str,
    qualifier: Optional[str] = None,
) -> str:
    candidates = _METHOD_TO_FRAMEWORKS.get(method, [])
    matching   = [fw for fw in candidates if fw in imported]

    if len(matching) == 1:
        return matching[0]
    if len(matching) > 1:
        if qualifier == "Assertions":
            for fw in ("junit5", "assertj"):
                if fw in matching: return fw
        if qualifier == "Assert":
            for fw in ("junit4", "testng"):
                if fw in matching: return fw
        return matching[0]
    if candidates:
        return candidates[0]
    return "unknown"


# =============================================================================
# FILE ANALYZER
# =============================================================================

def analyze_file(filepath: str) -> FileResult:
    result = FileResult(path=filepath)

    try:
        raw_bytes = Path(filepath).read_bytes()
        raw_str   = raw_bytes.decode("utf-8", errors="replace")
    except OSError as e:
        result.parse_error = str(e)
        return result

    result.raw_imports, result.frameworks_imported = \
        _detect_frameworks_from_imports(raw_str)
    result.has_assert_in_comments = _has_assert_in_comments(raw_str)

    try:
        tree = PARSER.parse(raw_bytes)
    except Exception as e:
        result.parse_error = f"tree-sitter parse error: {e}"
        return result

    if tree.root_node.has_error:
        result.parse_error = "syntax warnings (partial parse used)"

    root         = tree.root_node
    var_types    = _collect_local_var_types(root, raw_bytes)
    test_methods = _extract_test_methods(root, raw_bytes)
    helper_names = _find_helper_method_names(root, raw_bytes)

    result.test_methods   = test_methods
    result.helper_methods = sorted(helper_names)
    result.assert_calls   = _collect_assert_calls(
        root, raw_bytes,
        result.frameworks_imported,
        var_types,
        test_methods,
    )
    return result


def analyze_directory(root: str, pattern: str = "*Test*.java") -> list[FileResult]:
    return [analyze_file(str(p)) for p in Path(root).rglob(pattern)]


# =============================================================================
# JSON EXPORT
# =============================================================================

def export_json(results: list[FileResult], out_path: str = "ast_analysis.json"):
    output = []
    for r in results:

        test_method_call_ids = set()
        method_summary = []
        for tm in r.test_methods:
            if not tm.is_test:
                continue
            for c in tm.assert_calls:
                test_method_call_ids.add(id(c))
            method_summary.append({
                "name":       tm.name,
                "start_line": tm.start_line,
                "end_line":   tm.end_line,
                "assert_calls": [
                    {
                        "method":      c.method,
                        "line":        c.line,
                        "framework":   c.framework,
                        "in_trycatch": c.in_trycatch,
                        "source":      c.source,
                        "call_text":   c.call_text,
                    }
                    for c in tm.assert_calls
                ],
            })

        flat_calls = []
        for c in r.assert_calls:
            flat_calls.append({
                "method":            c.method,
                "line":              c.line,
                "framework":         c.framework,
                "in_trycatch":       c.in_trycatch,
                "source":            c.source,
                "call_text":         c.call_text,
                "test_method":       c.test_method,
                "test_method_start": c.test_method_start,
                "test_method_end":   c.test_method_end,
                "in_test_method":    id(c) in test_method_call_ids,
            })

        output.append({
            "file":                   Path(r.path).name,
            "path":                   r.path,
            "frameworks_imported":    list(r.frameworks_imported),
            "has_import":             r.has_import,
            "has_real_calls":         r.has_real_calls,
            "import_only":            r.import_only,
            "comment_only":           r.comment_only,
            "has_assert_in_comments": r.has_assert_in_comments,
            "total_assert_calls":     len(r.assert_calls),
            "helper_methods":         r.helper_methods,
            "parse_error":            r.parse_error,
            "assert_calls":           flat_calls,
            "test_methods":           method_summary,
        })

    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(output, f, indent=2)
    print(f"  Details: {out_path}")


# =============================================================================
# PREPROCESSING (.txt → .java)
# =============================================================================

def clean_java_code(content: str) -> str:
    content = re.sub(r'^```java\s*\n', '', content, flags=re.MULTILINE)
    content = re.sub(r'^```\s*\n',     '', content, flags=re.MULTILINE)
    content = re.sub(r'\n```\s*$',     '', content, flags=re.MULTILINE)
    content = re.sub(r'^```\s*$',      '', content, flags=re.MULTILINE)
    content = re.sub(r'^.*?(?=\s*(package|import|public|class|//|/\*))',
                     '', content, count=1, flags=re.DOTALL)
    return content.strip()


def extract_class_name(content: str) -> Optional[str]:
    m = re.search(r'public\s+class\s+(\w+)', content)
    return m.group(1) if m else None


def prepare_java_file(txt_path: Path) -> Optional[Path]:
    try:
        content = txt_path.read_text(encoding='utf-8')
    except Exception as e:
        print(f"    ⚠️  Error reading {txt_path.name}: {e}")
        return None

    cleaned    = clean_java_code(content)
    class_name = extract_class_name(cleaned) or txt_path.stem.replace('_prompt', '')

    java_dir  = txt_path.parent / "_java_files"
    java_dir.mkdir(exist_ok=True)
    java_file = java_dir / f"{class_name}.java"

    try:
        java_file.write_text(cleaned, encoding='utf-8')
        return java_file
    except Exception as e:
        print(f"   Error writing {java_file}: {e}")
        return None


def find_test_files(root_dir: str, patterns: list[str]) -> list[Path]:
    files = []
    root  = Path(root_dir)
    for pattern in patterns:
        for f in root.rglob(pattern):
            if "_java_files" not in f.parts:
                files.append(f)
    return sorted(set(files))


# =============================================================================
# STATS + REPORTING
# =============================================================================

def compute_stats(results: list) -> dict:
    total = len(results)

    files_with_calls   = sum(1 for r in results if r.has_real_calls)
    files_import_only  = sum(1 for r in results if r.import_only)
    files_comment_only = sum(1 for r in results if r.comment_only)
    files_no_assert    = sum(
        1 for r in results
        if not r.has_real_calls and not r.import_only and not r.comment_only
    )
    files_with_helpers = sum(1 for r in results if r.helper_methods)

    framework_call_counts: dict[str, int] = defaultdict(int)
    framework_file_counts: dict[str, int] = defaultdict(int)
    method_counts:         dict[str, int] = defaultdict(int)

    for r in results:
        frameworks_seen_in_file = set()
        for c in r.assert_calls:
            framework_call_counts[c.framework] += 1
            method_counts[c.method]            += 1
            frameworks_seen_in_file.add(c.framework)
        for fw in frameworks_seen_in_file:
            framework_file_counts[fw] += 1

    all_test_methods = [
        tm for r in results
        for tm in r.test_methods if tm.is_test
    ]
    total_test_methods = len(all_test_methods)

    test_assert_counts   = [len(tm.assert_calls) for tm in all_test_methods]
    methods_with_asserts = sum(1 for c in test_assert_counts if c > 0)
    methods_no_asserts   = total_test_methods - methods_with_asserts
    total_calls          = sum(test_assert_counts)
    min_asserts          = min(test_assert_counts) if test_assert_counts else 0
    max_asserts          = max(test_assert_counts) if test_assert_counts else 0
    avg_asserts          = total_calls / total_test_methods if total_test_methods > 0 else 0

    in_trycatch = sum(
        sum(1 for c in r.assert_calls if c.in_trycatch) for r in results
    )
    files_with_trycatch = sum(
        1 for r in results if any(c.in_trycatch for c in r.assert_calls)
    )
    soft = sum(1 for r in results if any(c.source == "soft" for c in r.assert_calls))

    return {
        "total_files":                  total,
        "files_with_assert_calls":      files_with_calls,
        "files_import_only":            files_import_only,
        "files_comment_only":           files_comment_only,
        "files_no_assert":              files_no_assert,
        "files_with_helpers":           files_with_helpers,
        "total_test_methods":           total_test_methods,
        "test_methods_with_asserts":    methods_with_asserts,
        "test_methods_without_asserts": methods_no_asserts,
        "total_assert_calls":           total_calls,
        "min_asserts_per_test_method":  min_asserts,
        "max_asserts_per_test_method":  max_asserts,
        "avg_asserts_per_test_method":  avg_asserts,
        "calls_in_trycatch":            in_trycatch,
        "files_with_trycatch_asserts":  files_with_trycatch,
        "files_with_soft_assertions":   soft,
        "framework_call_counts":        dict(framework_call_counts),
        "framework_file_counts":        dict(framework_file_counts),
        "method_breakdown":             dict(method_counts),
    }


def print_summary(stats: dict):
    total = stats['total_files']

    print(f"\n  📊 SUMMARY")
    print(f"  {'-'*66}")

    print(f"\n  FILE LEVEL  (total: {total})")
    print(f"  {'Category':<40} {'Count':>6}  {'%':>6}")
    print(f"  {'-'*55}")
    categories = [
        ("Files with assert calls",  "files_with_assert_calls"),
        ("Files with import only",   "files_import_only"),
        ("Files with comments only", "files_comment_only"),
        ("Files with no assert",     "files_no_assert"),
    ]
    for label, key in categories:
        cnt = stats[key]
        pct = 100 * cnt / total if total > 0 else 0
        print(f"  {label:<40} {cnt:>6}  {pct:>5.1f}%")
    subtotal = sum(stats[k] for _, k in categories)
    print(f"  {'-'*55}")
    print(f"  {'Total':<40} {subtotal:>6}")
    print(f"  Files with helper methods       : {stats['files_with_helpers']}")

    print(f"\n  @TEST METHOD LEVEL")
    print(f"  Total @Test methods             : {stats['total_test_methods']}")
    print(f"  @Test methods with asserts      : {stats['test_methods_with_asserts']}")
    print(f"  @Test methods without asserts   : {stats['test_methods_without_asserts']}")
    print(f"  ")
    print(f"  Total assert calls              : {stats['total_assert_calls']}")
    print(f"  Min asserts per @Test method    : {stats['min_asserts_per_test_method']}")
    print(f"  Max asserts per @Test method    : {stats['max_asserts_per_test_method']}")
    print(f"  Avg asserts per @Test method    : {stats['avg_asserts_per_test_method']:.2f}")
    print(f"  Calls inside try/catch          : {stats['calls_in_trycatch']}")
    print(f"  Files with assert in try/catch  : {stats['files_with_trycatch_asserts']}")
    print(f"  Files with soft assertions      : {stats['files_with_soft_assertions']}")

    if stats['framework_call_counts']:
        print(f"\n  Framework breakdown (calls | files using it):")
        for fw in sorted(stats['framework_call_counts'], key=lambda x: -stats['framework_call_counts'][x]):
            calls = stats['framework_call_counts'].get(fw, 0)
            files = stats['framework_file_counts'].get(fw, 0)
            print(f"    {fw:<20} {calls:>5} calls   {files:>5} files")

    if stats['method_breakdown']:
        print(f"\n  Top 10 assert methods:")
        for m, cnt in sorted(stats['method_breakdown'].items(), key=lambda x: -x[1])[:10]:
            print(f"    {m:<30} {cnt:>5} calls")
    print()


def generate_comparison_report(all_results: dict):
    print(f"\n{'='*70}")
    print(f"  COMPARISON ACROSS DATASETS")
    print(f"{'='*70}\n")

    rows = []
    for name in ["compiled_pre", "executed_pre", "detected_bre"]:
        if name not in all_results or not all_results[name]:
            continue
        s = all_results[name]["stats"]
        rows.append({
            "Dataset":            name,
            "Files":              s["total_files"],
            "w/ Calls":           s["files_with_assert_calls"],
            "Import Only":        s["files_import_only"],
            "No Assert":          s["files_no_assert"],
            "@Test Methods":      s["total_test_methods"],
            "w/ Asserts":         s["test_methods_with_asserts"],
            "w/o Asserts":        s["test_methods_without_asserts"],
            "Total Calls":        s["total_assert_calls"],
            "Min":                s["min_asserts_per_test_method"],
            "Max":                s["max_asserts_per_test_method"],
            "Avg":                f"{s['avg_asserts_per_test_method']:.2f}",
            "In Try/Catch":       s["calls_in_trycatch"],
            "Files w/ Try/Catch": s["files_with_trycatch_asserts"],
        })

    if not rows:
        print("  No data to compare.\n")
        return

    headers = list(rows[0].keys())
    widths  = [
        max(len(str(r.get(h, ""))) for r in rows + [{h: h}])
        for h in headers
    ]
    print("  " + " | ".join(h.ljust(w) for h, w in zip(headers, widths)))
    print("  " + "-+-".join("-" * w for w in widths))
    for row in rows:
        print("  " + " | ".join(str(row.get(h, "")).ljust(w) for h, w in zip(headers, widths)))
    print()


# =============================================================================
# CACHE — save/load all_results stats to survive kernel restarts
# =============================================================================

def save_results(all_results: dict):
    """Save all_results stats to JSON so they survive kernel restarts."""
    output_dir = Path(OUTPUT_DIR)
    output_dir.mkdir(parents=True, exist_ok=True)
    serializable = {
        name: data["stats"]
        for name, data in all_results.items()
        if data
    }
    with open(RESULTS_CACHE, "w") as f:
        json.dump(serializable, f, indent=2)
    print(f"  Results cached: {RESULTS_CACHE}")


def load_results(path: str = RESULTS_CACHE) -> dict:
    """Load cached stats — enough for plotting and comparison reports."""
    with open(path) as f:
        raw = json.load(f)
    return {name: {"stats": stats} for name, stats in raw.items()}


# =============================================================================
# DATASET RUNNER
# =============================================================================

def analyze_dataset(name: str, root_dir: str) -> Optional[dict]:
    print(f"\n{'='*70}")
    print(f"  Analyzing : {name}")
    print(f"  Location  : {root_dir}")
    print(f"{'='*70}\n")

    if not Path(root_dir).exists():
        print(f"    Directory does not exist, skipping.\n")
        return None

    files = find_test_files(root_dir, FILE_PATTERNS)
    if not files:
        print(f"    No test files found.\n")
        return None

    print(f"  Found {len(files)} source files")
    print(f"  Preparing Java files...\n")

    java_files = []
    for f in files:
        if f.suffix == '.txt':
            jf = prepare_java_file(f)
            if jf:
                java_files.append(jf)
        else:
            java_files.append(f)

    if not java_files:
        print(f"    No valid Java files after preprocessing.\n")
        return None

    print(f"  Analyzing {len(java_files)} Java files...\n")

    results = []
    for i, filepath in enumerate(java_files, 1):
        if i % 50 == 0:
            print(f"    Progress: {i}/{len(java_files)}")
        results.append(analyze_file(str(filepath)))

    for r in results:
        if not hasattr(r, 'test_methods') or r.test_methods is None:
            r.test_methods = []

    stats = compute_stats(results)
    stats.update({
        "dataset_name":   name,
        "root_dir":       root_dir,
        "file_count":     len(files),
        "analyzed_count": len(java_files),
    })

    print_summary(stats)
    return {"stats": stats, "results": results}


def save_json_reports(all_results: dict):
    output_dir = Path(OUTPUT_DIR)
    output_dir.mkdir(parents=True, exist_ok=True)

    for name, data in all_results.items():
        if not data:
            continue
        stats_file = output_dir / f"{name}_stats.json"
        with open(stats_file, "w") as f:
            json.dump(data["stats"], f, indent=2)
        print(f"  Stats  : {stats_file}")

        details_file = str(output_dir / f"{name}_details.json")
        export_json(data["results"], details_file)

    print()


# =============================================================================
# MAIN
# =============================================================================

def main():
    print(f"\n{'='*70}")
    print(f"  Java Assert Analysis Runner  (v3 — per-method tracking)")
    print(f"{'='*70}")

    all_results = {}
    for name, path in DATASETS.items():
        result = analyze_dataset(name, path)
        if result:
            all_results[name] = result

    if not all_results:
        print("\n  No datasets analyzed. Check your paths.\n")
        return None

    generate_comparison_report(all_results)
    save_json_reports(all_results)
    save_results(all_results)

    print(f"\n{'='*70}")
    print(f"   Analysis complete!")
    print(f"   Reports saved to : {OUTPUT_DIR}")
    print(f"   Cleaned .java    : each instance/_java_files/")
    print(f"{'='*70}\n")

    return all_results


# =============================================================================
# ENTRY POINT
# =============================================================================

# In a notebook cell, call:
#   all_results = main()
#
# After a kernel restart, reload without re-running analysis:
#   all_results = load_results()

if __name__ == "__main__":
    all_results = main()


  Java Assert Analysis Runner  (v3 — per-method tracking)

  Analyzing : compiled_pre
  Location  : /Volumes/Rachna-HD/ResultsDataset/Exp3LLMOutput/GPT4o/compiled_pre

  Found 971 source files
  Preparing Java files...

  Analyzing 971 Java files...

    Progress: 50/971
    Progress: 100/971
    Progress: 150/971
    Progress: 200/971
    Progress: 250/971
    Progress: 300/971
    Progress: 350/971
    Progress: 400/971
    Progress: 450/971
    Progress: 500/971
    Progress: 550/971
    Progress: 600/971
    Progress: 650/971
    Progress: 700/971
    Progress: 750/971
    Progress: 800/971
    Progress: 850/971
    Progress: 900/971
    Progress: 950/971

  📊 SUMMARY
  ------------------------------------------------------------------

  FILE LEVEL  (total: 971)
  Category                                  Count       %
  -------------------------------------------------------
  Files with assert calls                     904   93.1%
  Files with import only                       

In [27]:
for name in ["compiled_pre", "executed_pre", "detected_bre"]:
    s = all_results[name]["stats"]
    print(f"{name}: total={s['total_files']}, with_asserts={s['files_with_assert_calls']}, trycatch={s['files_with_trycatch_asserts']}")

compiled_pre: total=971, with_asserts=904, trycatch=697
executed_pre: total=418, with_asserts=381, trycatch=266
detected_bre: total=168, with_asserts=135, trycatch=130


shankey input:

// Bug Lifecycle Phases — Assert Usage (file counts)

// Compiled phase
Compiled [904] Has Asserts (Compiled)
Compiled [67] No Asserts (Compiled)

// Executed phase
Executed [381] Has Asserts (Executed)
Executed [37] No Asserts (Executed)

// Detected phase
Detected [135] Has Asserts (Detected)
Detected [33] No Asserts (Detected)

// Try/catch breakdown
Has Asserts (Compiled) [697] Try/Catch (Compiled)
Has Asserts (Compiled) [207] Normal (Compiled)

Has Asserts (Executed) [266] Try/Catch (Executed)
Has Asserts (Executed) [115] Normal (Executed)

Has Asserts (Detected) [130] Try/Catch (Detected)
Has Asserts (Detected) [5] Normal (Detected)

// Colors
:Compiled #4C72B0
:Executed #55A868
:Detected #C44E52